In [1]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /Users/gremy/Code/TSNN-1
tsnn module path: /Users/gremy/Code/TSNN-1/tsnn


In [2]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers



plt.style.use('ggplot')

In [ ]:
from dataclasses import dataclass
from torch import nn


In [34]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

Using device: mps


In [4]:
from typing import Dict

In [5]:
from tsnn.tstorch import models

In [6]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer
from sklearn.ensemble import HistGradientBoostingRegressor



# Summary

In [7]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [8]:
# We work with the following data.

In [9]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

5000 10 20 5


In [10]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [11]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "correl" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [12]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

In [13]:
effects = list(dic_data['correl0.1'].keys())
print(effects)

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']


In [14]:
# We will fix the above dataset for now.

In [15]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [16]:
# Let's list here all the models we wish to test on all the data.

In [17]:
def get_models():
    MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

    MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

    trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                            dropout=0.2, roll_y=True).to(device)
    #Note: using the MLP compression seems very bad..

    trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)

    trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)


    dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

    trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)

    trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)


    dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}
    
    return dic_models, dic_models_rollfalse

In [18]:
# For each model we also need to specify the option we will use to fit them.

In [19]:
dic_data['correl0.1']

{'d_lin': <tsnn.generators.generators.Generator at 0x13cdc4d10>,
 'd_cond': <tsnn.generators.generators.Generator at 0x14f0e4fd0>,
 'd_shift': <tsnn.generators.generators.Generator at 0x168146950>,
 'd_cs': <tsnn.generators.generators.Generator at 0x168146d50>,
 'd_cs_shift': <tsnn.generators.generators.Generator at 0x14f016a50>,
 'd_all': <tsnn.generators.generators.Generator at 0x168147190>}

In [20]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

# Function to create table for an effect

In [21]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [ ]:
def run_models(effect1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():
        
        z = dic_data[noise_level][effect1]
        z.get_dataloader(n_rolling=T1)

        dic_models, dic_models_rollfalse = get_models()


        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        records_train.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        records_train.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })
        


        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = ["lasso_full", "boosting"] + list(dic_models.keys())
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_lin0, table_test_lin0 = run_models('d_lin')

KeyboardInterrupt: 

In [ ]:
display(table_train_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [ ]:
display(table_test_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

In [30]:
# Saving the data

table_train_lin0.to_csv('table_train_lin.csv', index=True)
table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [31]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [32]:
dic_models, dic_models_rollfalse = get_models()

table_train_cond0, table_test_cond0 = run_models('d_cond')

Running → correl0.1 | MLP_global


100%|██████████| 20/20 [00:03<00:00,  6.33it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.1 | MLP_2D


100%|██████████| 20/20 [00:04<00:00,  4.14it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.1 | trans_1D_T4


100%|██████████| 20/20 [00:09<00:00,  2.20it/s]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.1 | trans_2D_TCTC


100%|██████████| 20/20 [00:24<00:00,  1.23s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.1 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:46<00:00,  2.32s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [33]:
display(table_train_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.1,0.032,0.077,0.302,0.287,0.280,0.489,0.486


In [34]:
display(table_test_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.1,-0.006,-0.012,-0.003,0.005,0.020,0.444,0.442


In [26]:
# Saving the data

#table_train_cond0.to_csv('table_train_cond.csv', index=True)
#table_test_cond0.to_csv('table_test_cond.csv', index=True)

In [ ]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_shift0, table_test_shift0 = run_models('d_shift')

In [38]:
display(table_train_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.018,0.014,0.025,0.031,0.027,0.077,0.050
correl0.03,0.018,0.016,0.106,0.106,0.105,0.112,0.107
correl0.05,0.058,0.038,0.147,0.136,0.149,0.162,0.150
correl0.1,0.150,0.081,0.290,0.324,0.302,0.334,0.311
correl0.3,0.303,0.193,0.658,0.829,0.695,0.764,0.724
correl0.5,0.307,0.233,0.806,0.946,0.863,0.937,0.896


In [39]:
display(table_test_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.020,0.017,0.032,0.024,-0.011,0.090,0.055
correl0.03,0.021,0.008,0.037,0.039,0.022,0.074,0.057
correl0.05,0.058,0.032,0.100,0.056,0.058,0.129,0.083
correl0.1,0.133,0.030,0.090,0.232,0.086,0.275,0.209
correl0.3,0.297,0.186,0.346,0.862,0.406,0.771,0.722
correl0.5,0.316,0.254,0.547,0.942,0.720,0.938,0.899


In [40]:
# Saving the data

table_train_shift0.to_csv('table_train_shift.csv', index=True)
table_test_shift0.to_csv('table_test_shift.csv', index=True)

In [41]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_cs0, table_test_cs0 = run_models('d_cs')

In [43]:
display(table_train_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.001,0.003,0.029,0.025,0.034,0.040,0.030
correl0.03,0.033,0.031,0.089,0.086,0.095,0.093,0.089
correl0.05,0.066,0.055,0.146,0.137,0.173,0.152,0.154
correl0.1,0.168,0.091,0.293,0.343,0.346,0.319,0.307
correl0.3,0.295,0.185,0.643,0.848,0.749,0.789,0.714
correl0.5,0.311,0.232,0.833,0.935,0.898,0.924,0.878


In [44]:
display(table_test_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.008,-0.007,0.018,0.002,0.044,0.023,0.007
correl0.03,0.027,0.011,0.039,0.013,0.127,0.042,0.053
correl0.05,0.055,0.019,0.068,0.034,0.209,0.054,0.065
correl0.1,0.135,0.037,0.157,0.258,0.426,0.171,0.151
correl0.3,0.312,0.186,0.397,0.869,0.804,0.818,0.768
correl0.5,0.315,0.250,0.724,0.936,0.926,0.941,0.926


In [45]:
# Saving the data

table_train_cs0.to_csv('table_train_cs.csv', index=True)
table_test_cs0.to_csv('table_test_cs.csv', index=True)

In [46]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [ ]:
dic_models, dic_models_rollfalse = get_models()

table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift')

In [48]:
display(table_train_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.001,-0.000,0.020,0.024,0.022,0.005,0.015
correl0.03,0.004,0.016,0.086,0.096,0.087,0.088,0.088
correl0.05,0.055,0.041,0.158,0.152,0.156,0.168,0.158
correl0.1,0.151,0.082,0.294,0.350,0.296,0.319,0.295
correl0.3,0.306,0.191,0.673,0.849,0.696,0.764,0.714
correl0.5,0.320,0.242,0.846,0.937,0.873,0.918,0.875


In [49]:
display(table_test_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.008,0.000,0.013,0.029,0.017,-0.009,-0.002
correl0.03,0.013,0.006,0.070,0.054,0.038,0.047,0.043
correl0.05,0.044,0.010,0.077,0.088,0.044,0.066,0.028
correl0.1,0.127,0.046,0.172,0.298,0.095,0.213,0.090
correl0.3,0.303,0.165,0.481,0.870,0.398,0.744,0.567
correl0.5,0.307,0.243,0.739,0.938,0.798,0.919,0.857


In [50]:
# Saving the data

table_train_csshift0.to_csv('table_train_csshift.csv', index=True)
table_test_csshift0.to_csv('table_test_csshift.csv', index=True)

In [51]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [25]:
def run_models_all_effect(noise_level1, dic_models, dic_models_rollfalse):

    z = dic_data[noise_level1]["d_all"]
    z.get_dataloader(n_rolling=T1)

    lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                            verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
    lasso_full.fit(z.train)
    boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
    boost_model.fit(z.train)

    list_models = [lasso_full, boost_model]


    for model_key in dic_models.keys():                   
        print(f"Running → {noise_level1} | {model_key}")

        if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1, roll_y=True)
        else:
            z.get_dataloader(n_rolling=T1)

        if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            lr=0.001/2
        else:
            lr=0.001

        epochs = 20
        if noise_level1 in ['correl0.01', 'correl0.03']:
            epochs = 40

        model = dic_models[model_key]

        # Model
        if noise_level1 == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1)
            model = dic_models_rollfalse[model_key]
            epochs = 60
            lr=0.0001

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

        
        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

        list_models.append(wrapper)

    comp = benchmark_comparison.Comparator(models=list_models, model_names=["lasso_full", "boosting"] + list(dic_models.keys()))

    corr_train = comp.correl(z, mode="train", return_values=True)
    corr_test  = comp.correl(z, mode="test",  return_values=True)

    return corr_train, corr_test

    

## Run on all noise levels

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_1, all_effects_test_1 = run_models_all_effect("correl0.01", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_3, all_effects_test_3 = run_models_all_effect("correl0.03", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_5, all_effects_test_5 = run_models_all_effect("correl0.05", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_10, all_effects_test_10 = run_models_all_effect("correl0.1", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_30, all_effects_test_30 = run_models_all_effect("correl0.3", dic_models, dic_models_rollfalse)

In [ ]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_50, all_effects_test_50 = run_models_all_effect("correl0.5", dic_models, dic_models_rollfalse)

In [32]:
display(all_effects_train_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.029,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.020,0.447,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.013,0.445,-0.004,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.006,0.449,-0.001,0.005,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.017,0.446,-0.000,-0.002,0.002,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.009,0.450,0.003,-0.002,-0.001,0.001,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.107,0.007,-0.000,0.001,-0.002,0.007,0.011,nan,nan,nan,nan,nan,nan
boosting,0.259,0.005,0.010,0.002,0.001,0.009,-0.010,0.305,nan,nan,nan,nan,nan
MLP_global,0.989,0.029,0.020,0.012,0.006,0.016,0.010,0.107,0.256,nan,nan,nan,nan
MLP_2D,0.977,0.031,0.020,0.012,0.008,0.019,0.010,0.105,0.254,0.967,nan,nan,nan


In [33]:
display(all_effects_test_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.029,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.003,0.450,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.028,0.446,0.003,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.009,0.435,0.001,-0.012,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.016,0.449,-0.004,0.003,-0.011,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.009,0.457,-0.004,0.006,-0.001,0.019,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.001,0.019,0.006,0.001,-0.005,0.009,0.030,nan,nan,nan,nan,nan,nan
boosting,0.007,-0.005,0.014,-0.018,-0.003,0.000,-0.004,0.128,nan,nan,nan,nan,nan
MLP_global,0.012,0.042,0.030,0.018,0.004,0.017,0.024,0.082,0.037,nan,nan,nan,nan
MLP_2D,-0.005,0.032,0.003,0.010,0.014,0.033,0.012,0.035,0.009,0.308,nan,nan,nan


In [34]:
display(all_effects_train_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.097,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.046,0.451,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.045,0.449,0.006,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.045,0.447,-0.001,-0.003,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.038,0.445,-0.004,-0.003,0.006,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.045,0.458,0.007,0.009,0.007,0.007,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.115,0.027,0.005,0.016,0.019,0.007,0.013,nan,nan,nan,nan,nan,nan
boosting,0.262,0.031,0.013,0.020,0.012,0.009,0.015,0.330,nan,nan,nan,nan,nan
MLP_global,0.989,0.096,0.047,0.044,0.044,0.038,0.043,0.112,0.258,nan,nan,nan,nan
MLP_2D,0.977,0.094,0.043,0.042,0.044,0.038,0.045,0.114,0.257,0.967,nan,nan,nan


In [35]:
display(all_effects_test_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.101,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.043,0.457,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.037,0.439,-0.010,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.043,0.446,0.011,0.000,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.061,0.449,0.009,0.001,-0.001,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.042,0.448,0.012,-0.016,0.004,-0.002,nan,nan,nan,nan,nan,nan,nan
lasso_full,-0.001,0.017,0.009,0.007,0.013,0.002,0.006,nan,nan,nan,nan,nan,nan
boosting,0.005,0.020,0.008,0.006,0.024,0.004,0.003,0.199,nan,nan,nan,nan,nan
MLP_global,0.031,0.093,0.056,0.003,0.051,0.042,0.058,0.065,0.017,nan,nan,nan,nan
MLP_2D,0.019,0.050,0.021,0.011,0.032,0.008,0.041,0.036,0.007,0.200,nan,nan,nan


In [36]:
display(all_effects_train_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.160,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.074,0.451,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.076,0.457,0.006,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.071,0.453,0.002,0.019,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.072,0.442,0.003,-0.000,-0.005,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.070,0.453,0.008,0.004,0.006,-0.001,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.122,0.050,0.034,-0.002,0.042,0.035,0.003,nan,nan,nan,nan,nan,nan
boosting,0.263,0.042,0.028,0.014,0.019,0.018,0.017,0.351,nan,nan,nan,nan,nan
MLP_global,0.980,0.159,0.074,0.075,0.068,0.072,0.069,0.121,0.260,nan,nan,nan,nan
MLP_2D,0.916,0.158,0.078,0.070,0.073,0.070,0.066,0.117,0.243,0.898,nan,nan,nan


In [37]:
display(all_effects_test_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.156,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.059,0.452,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.069,0.444,-0.011,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.066,0.447,0.006,0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.084,0.447,0.011,0.001,-0.015,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.072,0.450,0.007,-0.003,0.001,0.007,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.008,0.040,0.031,0.007,0.026,0.025,-0.000,nan,nan,nan,nan,nan,nan
boosting,0.004,0.018,-0.003,0.004,0.018,0.001,0.021,0.146,nan,nan,nan,nan,nan
MLP_global,0.007,0.066,0.025,0.005,0.050,0.041,0.028,0.041,0.018,nan,nan,nan,nan
MLP_2D,0.005,0.068,0.042,0.001,0.058,0.031,0.021,0.040,0.025,0.079,nan,nan,nan


In [38]:
display(all_effects_train_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.310,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.149,0.468,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.141,0.460,0.014,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.143,0.460,0.019,0.013,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.138,0.466,0.026,0.031,0.009,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.149,0.469,0.029,0.011,0.027,0.019,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.124,0.128,0.085,0.015,0.092,0.041,0.064,nan,nan,nan,nan,nan,nan
boosting,0.255,0.080,0.038,0.040,0.038,0.033,0.035,0.346,nan,nan,nan,nan,nan
MLP_global,0.978,0.307,0.146,0.139,0.146,0.137,0.146,0.124,0.252,nan,nan,nan,nan
MLP_2D,0.918,0.328,0.165,0.131,0.149,0.152,0.166,0.120,0.230,0.900,nan,nan,nan


In [39]:
display(all_effects_test_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.308,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.150,0.466,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.140,0.476,0.029,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.152,0.470,0.020,0.029,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.147,0.466,0.017,0.025,0.031,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.131,0.456,0.021,0.018,0.025,0.010,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.041,0.110,0.067,0.003,0.078,0.047,0.062,nan,nan,nan,nan,nan,nan
boosting,0.005,0.029,0.024,0.007,0.011,0.012,0.016,0.225,nan,nan,nan,nan,nan
MLP_global,0.044,0.148,0.102,0.012,0.079,0.086,0.068,0.065,0.037,nan,nan,nan,nan
MLP_2D,0.080,0.265,0.159,0.021,0.115,0.143,0.181,0.050,0.024,0.095,nan,nan,nan


In [40]:
display(all_effects_train_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.690,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.385,0.565,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.390,0.565,0.148,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.390,0.564,0.145,0.151,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.399,0.571,0.157,0.153,0.151,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.393,0.573,0.154,0.156,0.156,0.158,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.219,0.295,0.180,0.092,0.186,0.182,0.195,nan,nan,nan,nan,nan,nan
boosting,0.265,0.195,0.108,0.106,0.110,0.108,0.121,0.611,nan,nan,nan,nan,nan
MLP_global,0.977,0.690,0.388,0.382,0.390,0.402,0.395,0.221,0.261,nan,nan,nan,nan
MLP_2D,0.940,0.752,0.436,0.368,0.437,0.451,0.444,0.240,0.252,0.925,nan,nan,nan


In [41]:
display(all_effects_test_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.684,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.382,0.563,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.393,0.568,0.149,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.379,0.563,0.142,0.157,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.396,0.573,0.154,0.159,0.145,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.399,0.581,0.155,0.159,0.166,0.168,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.188,0.278,0.173,0.066,0.185,0.184,0.184,nan,nan,nan,nan,nan,nan
boosting,0.116,0.169,0.118,0.036,0.113,0.094,0.120,0.615,nan,nan,nan,nan,nan
MLP_global,0.290,0.433,0.295,0.124,0.238,0.299,0.276,0.161,0.112,nan,nan,nan,nan
MLP_2D,0.527,0.779,0.489,0.229,0.483,0.504,0.512,0.255,0.159,0.412,nan,nan,nan


In [42]:
display(all_effects_train_50.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.844,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.580,0.683,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.574,0.681,0.332,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.576,0.683,0.333,0.331,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.570,0.679,0.331,0.328,0.328,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.574,0.681,0.331,0.332,0.335,0.325,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.262,0.304,0.219,0.153,0.212,0.214,0.239,nan,nan,nan,nan,nan,nan
boosting,0.268,0.237,0.159,0.156,0.160,0.156,0.177,0.743,nan,nan,nan,nan,nan
MLP_global,0.984,0.849,0.581,0.566,0.585,0.577,0.583,0.259,0.259,nan,nan,nan,nan
MLP_2D,0.959,0.895,0.627,0.559,0.622,0.615,0.626,0.279,0.257,0.953,nan,nan,nan


In [43]:
display(all_effects_test_50.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.849,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.586,0.690,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.582,0.682,0.332,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.582,0.689,0.344,0.339,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.582,0.687,0.347,0.339,0.337,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.587,0.691,0.349,0.337,0.345,0.342,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.260,0.308,0.225,0.143,0.220,0.232,0.240,nan,nan,nan,nan,nan,nan
boosting,0.208,0.245,0.169,0.123,0.175,0.184,0.192,0.792,nan,nan,nan,nan,nan
MLP_global,0.653,0.774,0.501,0.381,0.597,0.561,0.621,0.262,0.210,nan,nan,nan,nan
MLP_2D,0.758,0.896,0.659,0.451,0.654,0.644,0.672,0.288,0.228,0.762,nan,nan,nan


In [45]:
# Let's also save all the tables above.
all_effects_train_1.to_csv('all_effects_train_1.csv', index=True)
all_effects_test_1.to_csv('all_effects_test_1.csv', index=True)

all_effects_train_3.to_csv('all_effects_train_3.csv', index=True)
all_effects_test_3.to_csv('all_effects_test_3.csv', index=True)

all_effects_train_5.to_csv('all_effects_train_5.csv', index=True)
all_effects_test_5.to_csv('all_effects_test_5.csv', index=True)

all_effects_train_10.to_csv('all_effects_train_10.csv', index=True)
all_effects_test_10.to_csv('all_effects_test_10.csv', index=True)

all_effects_train_30.to_csv('all_effects_train_30.csv', index=True)
all_effects_test_30.to_csv('all_effects_test_30.csv', index=True)

all_effects_train_50.to_csv('all_effects_train_50.csv', index=True)
all_effects_test_50.to_csv('all_effects_test_50.csv', index=True)

# Testing sparsity

In [21]:
# To test sparsity let's write a function that runs one model on all the effects and noise levels.
# Then we can run on the model with and without sparsity.

In [22]:
def keep_topk_per_row(x, k=2):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out


In [23]:
effects

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']

In [32]:
def test_sparsity_all_correl_effets(sparsity=False):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():          
        for effect in ['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']:                   
            print(f"Running → {noise_level} | {effect}")

            z = dic_data[noise_level][effect]
            z.get_dataloader(n_rolling=T1, roll_y=True)
            
            lr=0.001/2
            roll_y=True
            
            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40


            # Model
            if noise_level == 'correl0.01':
                z.get_dataloader(n_rolling=T1)
                roll_y=False
                epochs = 60
                lr=0.0001

            # Model
            if sparsity:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=keep_topk_per_row, 
                                                                               roll_y=roll_y).to(device)
            else:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                                roll_y=roll_y).to(device)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer)

            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "effect": effect,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "effect": effect,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="effect", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="effect", values="test_corr_optimal")    

    # Sort columns in logical order
    col_order = ["d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"]
    train_model = train_pivot[col_order]
    test_model  = test_pivot[col_order]

    return train_model, test_model

In [25]:
table_train_no_sparsity, table_test_no_sparsity = test_sparsity_all_correl_effets()

Running → correl0.05 | d_lin


100%|██████████| 20/20 [00:25<00:00,  1.26s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_cond


100%|██████████| 20/20 [00:24<00:00,  1.20s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_shift


...

100%|██████████| 20/20 [00:23<00:00,  1.20s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_cs_shift


100%|██████████| 20/20 [00:23<00:00,  1.20s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_all


100%|██████████| 20/20 [00:23<00:00,  1.20s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [26]:
table_train_with_sparsity, table_test_with_sparsity = test_sparsity_all_correl_effets(sparsity=True)

Running → correl0.05 | d_lin


100%|██████████| 20/20 [00:24<00:00,  1.23s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_cond


100%|██████████| 20/20 [00:24<00:00,  1.23s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_shift


...

100%|██████████| 20/20 [00:24<00:00,  1.23s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_cs_shift


100%|██████████| 20/20 [00:24<00:00,  1.22s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Running → correl0.05 | d_all


100%|██████████| 20/20 [00:24<00:00,  1.22s/it]
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/gremy/Code/TSNN-1/tsnn-env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


In [27]:
display(table_train_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.05,0.653,0.332,0.397,0.491,0.046,0.367


In [29]:
display(table_train_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.05,0.557,0.299,0.442,0.562,0.071,0.336


In [30]:
display(table_test_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.05,0.637,0.306,0.399,0.494,-0.013,0.362


In [31]:
display(table_test_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.05,0.546,0.276,0.440,0.565,0.015,0.315


In [ ]:
# To store the tables above

table_train_no_sparsity.to_csv('table_train_no_sparsity.csv', index=True)
table_test_no_sparsity.to_csv('table_test_no_sparsity.csv', index=True)

table_train_with_sparsity.to_csv('table_train_with_sparsity.csv', index=True)
table_test_with_sparsity.to_csv('table_test_with_sparsity.csv', index=True)